# Surf the OpenAI Cookbook for Your Repositories

Finding the right cookbook recipe for your project can be time-consuming when there are hundreds of examples. This notebook shows how to use **OpenAI embeddings** to automatically discover the most relevant cookbook entries for any GitHub repository.

Given one or more GitHub repository URLs, the notebook:

1. Fetches each repository's description, README excerpt, topics, and primary language from the GitHub API.
2. Embeds every entry in the [OpenAI Cookbook registry](https://github.com/openai/openai-cookbook/blob/main/registry.yaml) using `text-embedding-3-small`.
3. Embeds a summary of your repository using the same model.
4. Ranks cookbook entries by cosine similarity and returns personalized recommendations.

**Prerequisites**

| Variable | Description |
|----------|-------------|
| `OPENAI_API_KEY` | Your OpenAI API key (required) |
| `GITHUB_TOKEN` | A [personal access token](https://github.com/settings/tokens) with `public_repo` scope (optional — raises the GitHub API rate limit from 60 to 5,000 requests/hour) |

## 1. Install dependencies

In [ ]:
!pip install openai requests pyyaml numpy -q

## 2. Setup

In [ ]:
import base64
import os
import re
from collections import defaultdict

import numpy as np
import requests
import yaml
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Optional: set a GitHub token to avoid hitting the unauthenticated rate limit (60 req/hr)
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

EMBEDDING_MODEL = "text-embedding-3-small"
COOKBOOK_REGISTRY_URL = (
    "https://raw.githubusercontent.com/openai/openai-cookbook/main/registry.yaml"
)

## 3. Load the cookbook registry

The `registry.yaml` file lists every entry in the OpenAI Cookbook along with its title, description, and tags.
We fetch it directly from GitHub so the notebook works as a standalone script regardless of where it runs.

In [ ]:
def load_registry(url: str = COOKBOOK_REGISTRY_URL) -> list[dict]:
    """Fetch and parse the cookbook registry YAML."""
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    entries = yaml.safe_load(resp.text)
    # Keep only valid dict entries (skip any YAML comments / nulls)
    return [e for e in entries if isinstance(e, dict)]


registry = load_registry()
print(f"Loaded {len(registry)} cookbook entries.")

## 4. Embed all cookbook entries

We concatenate each entry's title, description, and tags into a single string and embed the result.
Batching the requests keeps the API call count low — a single call covers up to 2,048 inputs.

In [ ]:
def entry_to_text(entry: dict) -> str:
    """Combine title, description, and tags into a searchable string."""
    parts = [entry.get("title", "")]
    if entry.get("description"):
        parts.append(entry["description"])
    tags = entry.get("tags") or []
    if tags:
        parts.append("tags: " + ", ".join(tags))
    return " | ".join(parts)


def embed_texts(
    texts: list[str],
    model: str = EMBEDDING_MODEL,
    batch_size: int = 100,
) -> np.ndarray:
    """Embed a list of strings in batches and return an (N, D) float32 array."""
    all_embeddings: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = [t.replace("\n", " ").strip() for t in texts[i : i + batch_size]]
        response = client.embeddings.create(input=batch, model=model)
        all_embeddings.extend([d.embedding for d in response.data])
    return np.array(all_embeddings, dtype=np.float32)


print("Embedding cookbook entries — this may take a few seconds…")
entry_texts = [entry_to_text(e) for e in registry]
entry_embeddings = embed_texts(entry_texts)
print(f"Done. Embedding matrix shape: {entry_embeddings.shape}")

## 5. Fetch repository information

We use the GitHub REST API to gather each repository's description, primary language, topics,
and the first 3,000 characters of its README. Stripping markdown syntax from the README keeps
the embedding focused on semantic content rather than formatting noise.

In [ ]:
def _github_headers() -> dict:
    headers = {"Accept": "application/vnd.github+json"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return headers


def _strip_markdown(text: str) -> str:
    """Lightly strip common markdown formatting."""
    text = re.sub(r"!\[.*?\]\(.*?\)", "", text)   # images
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)  # links
    text = re.sub(r"#+\s*", "", text)               # headings
    text = re.sub(r"`{1,3}[^`]*`{1,3}", "", text)   # inline / fenced code
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def fetch_repo_info(repo_url: str) -> dict:
    """
    Fetch metadata for a GitHub repository.

    Args:
        repo_url: Full URL (``https://github.com/owner/repo``) or short form
                  ``owner/repo``.

    Returns:
        Dictionary with keys: full_name, description, language, topics,
        readme_excerpt, stars, url.
    """
    repo_url = repo_url.strip().rstrip("/")
    match = re.search(r"github\.com/([^/]+/[^/]+)", repo_url)
    if match:
        owner_repo = match.group(1).rstrip(".git")
    elif re.match(r"^[^/]+/[^/]+$", repo_url):
        owner_repo = repo_url
    else:
        raise ValueError(f"Cannot parse repository identifier: {repo_url!r}")

    headers = _github_headers()
    base_url = f"https://api.github.com/repos/{owner_repo}"

    meta_resp = requests.get(base_url, headers=headers, timeout=15)
    meta_resp.raise_for_status()
    data = meta_resp.json()

    # README — optional
    readme_text = ""
    readme_resp = requests.get(f"{base_url}/readme", headers=headers, timeout=15)
    if readme_resp.status_code == 200:
        raw_content = readme_resp.json().get("content", "")
        try:
            decoded = base64.b64decode(raw_content).decode("utf-8", errors="ignore")
            readme_text = _strip_markdown(decoded)[:3000]
        except Exception:
            pass

    return {
        "full_name": data.get("full_name", owner_repo),
        "description": data.get("description") or "",
        "language": data.get("language") or "",
        "topics": data.get("topics") or [],
        "readme_excerpt": readme_text,
        "stars": data.get("stargazers_count", 0),
        "url": data.get("html_url", f"https://github.com/{owner_repo}"),
    }

## 6. Rank cookbook entries by relevance

We build a text summary of the repository, embed it, and then compute the cosine similarity
between that embedding and every pre-computed cookbook entry embedding.

In [ ]:
def repo_to_text(info: dict) -> str:
    """Build a compact text summary of a repository for embedding."""
    parts: list[str] = []
    if info["description"]:
        parts.append(info["description"])
    if info["language"]:
        parts.append(f"primary language: {info['language']}")
    if info["topics"]:
        parts.append("topics: " + ", ".join(info["topics"]))
    if info["readme_excerpt"]:
        # Use first 1,000 chars of the cleaned README as an extra signal
        parts.append(info["readme_excerpt"][:1000])
    return " | ".join(parts) if parts else info["full_name"]


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D vectors."""
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / norm) if norm > 0 else 0.0


def find_relevant_entries(
    repo_url: str,
    top_n: int = 10,
) -> tuple[list[dict], dict]:
    """
    Return the *top_n* most relevant cookbook entries for a GitHub repository.

    Args:
        repo_url: GitHub repository URL or ``owner/repo`` shorthand.
        top_n:    Number of recommendations to return.

    Returns:
        A tuple of (results, repo_info) where *results* is a list of dicts
        with ``entry`` and ``score`` keys sorted by descending similarity.
    """
    info = fetch_repo_info(repo_url)
    print(f"\nRepository : {info['full_name']}")
    print(f"Description: {info['description'] or '(none)'}")
    print(f"Language   : {info['language'] or '(unknown)'}")
    print(f"Topics     : {', '.join(info['topics']) if info['topics'] else '(none)'}")

    summary = repo_to_text(info)
    repo_emb = embed_texts([summary])[0]

    scores = [
        cosine_similarity(repo_emb, entry_emb)
        for entry_emb in entry_embeddings
    ]
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    results = [
        {"entry": registry[idx], "score": score}
        for idx, score in ranked[:top_n]
    ]
    return results, info


def display_recommendations(results: list[dict], repo_info: dict) -> None:
    """Pretty-print cookbook recommendations for a repository."""
    print(f"\nTop {len(results)} cookbook recommendations for {repo_info['full_name']}:\n")
    print(f"{'#':<4} {'Score':<8} Title")
    print("-" * 80)
    for rank, item in enumerate(results, start=1):
        entry = item["entry"]
        score = item["score"]
        title = entry.get("title", "(no title)")
        slug = entry.get("slug") or entry.get("path", "")
        cookbook_url = f"https://cookbook.openai.com/{slug}"
        description = entry.get("description", "")
        tags = ", ".join(entry.get("tags") or [])

        print(f"{rank:<4} {score:.4f}  {title}")
        if description:
            print(f"          {description}")
        if tags:
            print(f"          Tags: {tags}")
        print(f"          {cookbook_url}\n")

## 7. Try it — single repository

Set `REPO_URL` to any public GitHub repository and run the cell to get personalized cookbook recommendations.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/openai/openai-python"  # ← change this
TOP_N = 10                                             # number of recommendations
# ──────────────────────────────────────────────────────────────────────────────

results, repo_info = find_relevant_entries(REPO_URL, top_n=TOP_N)
display_recommendations(results, repo_info)

## 8. Try it — scan all repositories for a GitHub user

If you want to discover cookbook content across *all* of your repositories at once,
this section fetches your most-starred public repos, gets recommendations for each,
and then **aggregates** the similarity scores to surface the highest-priority entries
across your entire portfolio.

In [ ]:
def get_user_repos(username: str, max_repos: int = 20) -> list[str]:
    """
    Return URLs for a GitHub user's most-starred public, non-forked repositories.

    Args:
        username:  GitHub username.
        max_repos: Maximum number of repos to return.
    """
    headers = _github_headers()
    url = f"https://api.github.com/users/{username}/repos"
    params = {"type": "public", "sort": "stars", "per_page": max_repos}
    resp = requests.get(url, headers=headers, params=params, timeout=15)
    resp.raise_for_status()
    return [r["html_url"] for r in resp.json() if not r.get("fork")]

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
GITHUB_USERNAME = "openai"  # ← change to your GitHub username
MAX_REPOS = 5               # repositories to scan
TOP_N_PER_REPO = 5          # recommendations retrieved per repo before aggregation
TOP_N_FINAL = 10            # final aggregated recommendations to display
# ──────────────────────────────────────────────────────────────────────────────

repo_urls = get_user_repos(GITHUB_USERNAME, max_repos=MAX_REPOS)
print(f"Found {len(repo_urls)} repositories for @{GITHUB_USERNAME}:")
for url in repo_urls:
    print(f"  {url}")

In [ ]:
# Build a slug → index lookup so we can aggregate by identity
slug_to_idx = {
    (e.get("slug") or e.get("path", str(i))): i
    for i, e in enumerate(registry)
}

aggregate_scores: dict[int, float] = defaultdict(float)

for url in repo_urls:
    try:
        res, _ = find_relevant_entries(url, top_n=TOP_N_PER_REPO)
        for item in res:
            entry = item["entry"]
            key = entry.get("slug") or entry.get("path", "")
            idx = slug_to_idx.get(key)
            if idx is not None:
                aggregate_scores[idx] += item["score"]
    except Exception as exc:
        print(f"Skipping {url}: {exc}")

ranked_indices = sorted(aggregate_scores, key=aggregate_scores.__getitem__, reverse=True)

print(f"\nTop {TOP_N_FINAL} cookbook entries across all @{GITHUB_USERNAME} repositories:\n")
print(f"{'#':<4} {'Agg. Score':<12} Title")
print("-" * 80)
for rank, idx in enumerate(ranked_indices[:TOP_N_FINAL], start=1):
    entry = registry[idx]
    score = aggregate_scores[idx]
    title = entry.get("title", "(no title)")
    slug = entry.get("slug") or entry.get("path", "")
    cookbook_url = f"https://cookbook.openai.com/{slug}"
    print(f"{rank:<4} {score:<12.4f} {title}")
    print(f"             {cookbook_url}")